## Analysis Objective

The objective of this analysis is to evaluate retail performance from an omnichannel business perspective.

Rather than analyzing overall sales in isolation, the focus is on channel-driven profitability, returns, customer behavior, and operational insights that can support business decision-making.

### Phase A: Data Loading & Validation

- Purpose: Ensure cleaned data is ready for analysis

- Load cleaned dataset

- Validate schema and row counts

- Confirm presence of engineered features

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("/Users/hepigediya/Desktop/retail-omnichannel-analytics/data/processed/cleaned_retail_sales.csv")

In [3]:
df.shape

(1027017, 17)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1027017 entries, 0 to 1027016
Data columns (total 17 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   invoice_no             1027017 non-null  object 
 1   stock_code             1027017 non-null  object 
 2   description            1027017 non-null  object 
 3   quantity               1027017 non-null  int64  
 4   invoice_date           1027017 non-null  object 
 5   unit_price             1027017 non-null  float64
 6   customer_id            797815 non-null   float64
 7   country                1027017 non-null  object 
 8   is_return              1027017 non-null  bool   
 9   is_cancellation        1027017 non-null  bool   
 10  is_anonymous_customer  1027017 non-null  bool   
 11  sales_amount           1027017 non-null  float64
 12  order_date             1027017 non-null  object 
 13  order_month            1027017 non-null  int64  
 14  order_year        

In [5]:
df['invoice_date'] = pd.to_datetime(df['invoice_date'], errors='coerce')

In [6]:
df['invoice_date'].dtype

dtype('<M8[ns]')

In [7]:
df.head()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,is_return,is_cancellation,is_anonymous_customer,sales_amount,order_date,order_month,order_year,is_bulk_transaction,sales_channel
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,False,False,83.4,2009-12-01,12,2009,False,NaN
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,False,False,81.0,2009-12-01,12,2009,False,NaN
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,False,False,81.0,2009-12-01,12,2009,False,NaN
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,False,False,100.8,2009-12-01,12,2009,False,NaN
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,False,False,30.0,2009-12-01,12,2009,False,NaN


 ### Phase B: Channel Derivation (Core Step)

- Purpose: Establish omnichannel structure

- Derive sales_channel using invoice-level behavior

- Document assumptions and thresholds

- Validate channel distribution

- This phase defines the analytical foundation of the project.

In [8]:
""" Aggregate at Invoice Level
first move from line-item level → invoice level. """

invoice_summary = (
    df.groupby('invoice_no')
      .agg(
          total_quantity=('quantity', 'sum'),
          total_sales=('sales_amount', 'sum')
      )
      .reset_index()
)


In [9]:
# Channel derivation rule
invoice_summary['derived_channel'] = np.where(
    invoice_summary['total_quantity'] >= 20,
    'Offline',
    'Online'
)

In [10]:
invoice_summary['derived_channel'].value_counts()

derived_channel
Offline    35929
Online     12440
Name: count, dtype: int64

In [11]:
df = df.merge(
    invoice_summary[['invoice_no', 'derived_channel']],
    on='invoice_no',
    how='left'
)

In [12]:
df = df.rename(columns={'derived_channel': 'sales_channel'})

In [13]:
# Remove placeholder channel column from cleaning phase
df = df.drop(columns=['sales_channel'])

In [14]:
df = df.merge(
    invoice_summary[['invoice_no', 'derived_channel']],
    on='invoice_no',
    how='left'
)

df = df.rename(columns={'derived_channel': 'sales_channel'})

In [15]:
df['sales_channel'].value_counts()

sales_channel
Offline    996526
Online      30491
Name: count, dtype: int64

### Phase C: Channel-Level Revenue Analysis

- Purpose: Understand where money is coming from

- Total revenue by channel

- Revenue contribution percentage

- Revenue trend comparison by channel

- Business question:

    Which channel drives the majority of sales, and how stable is it?

### Phase D: Returns Impact by Channel

- Purpose: Identify profitability risk

- Return rate by channel

- Revenue lost due to returns

- Products with high return concentration by channel

- Business question:

    Which channel is more operationally expensive due to returns?

### Phase E: Customer Behavior by Channel

- Purpose: Evaluate customer value

- Number of customers per channel

- Anonymous vs identified customer impact

- Repeat vs one-time behavior (where applicable)

- Business question:

    Which channel produces more valuable and identifiable customers?

### Phase F: Product Performance by Channel

- Purpose: Optimize assortment and inventory

- Top products by revenue per channel

- High-volume vs high-return products

- Channel-specific product preferences

- Business question:

    Should the same products be pushed across both channels?

### Phase G: Time-Based Channel Trends

- Purpose: Detect seasonality and risk

- Monthly revenue trends by channel

- Peak and low periods

- Channel volatility comparison

- Business question:

    Which channel provides more predictable revenue over time?

### Phase H: Geographic Overlay (Optional Enhancement)

- Purpose: Add market-level context

- Country-wise channel performance

- Revenue concentration risk

- Channel dominance by geography

- Business question:

    Does channel performance vary by market?

### Phase I: Business Insights & Recommendations

- Purpose: Convert analysis into action

- Key findings summary

- Channel investment recommendations

- Risk areas and optimization opportunities